In [ ]:
import pandas as pd 
df = pd.read_csv("../Data/AllCities_combined_data.csv")
df

In [ ]:
df.info()


df.describe()

In [ ]:
df['City'].unique()

In [ ]:
import numpy as np
import pandas as pd

# ---- formula config -------------------------------------------------
POLLUTANTS    = ['NO2', 'CO', 'SO2', 'O3', 'AerosolIndex']   # 5 MPSI terms
WINTER_MONTHS = {10, 11, 12, 1, 2}                            # W_c : Oct - Feb
WINTER_CITIES = ['Delhi', 'Loni', 'Noida', 'Gurugram', 'Lahore', 'Faisalabad',
                 'Dhaka', 'Hanoi', 'Dushanbe', 'Hotan', 'Kashgar']
MPSI_THRESHOLD, ELEVATED_THRESHOLD, MIN_ELEVATED = 1.0, 1.0, 2
MIN_MONTHS_PER_YEAR = 8      # flag years too short for a stable baseline

# ---- 1. weekly long -> monthly wide ---------------------------------
sub = df[df['City'].isin(WINTER_CITIES) & df['Gas'].isin(POLLUTANTS)].copy()
missing = sorted(set(WINTER_CITIES) - set(sub['City'].unique()))
if missing:
    print('WARNING - not found in df:', missing)

monthly = (sub.groupby(['City', 'Country', 'Year', 'Month', 'Gas'])['Mean_Value']
              .mean()
              .unstack('Gas')
              .reset_index()
              .dropna(subset=POLLUTANTS))
monthly['date'] = (monthly['Year'].astype(str) + '-' +
                   monthly['Month'].astype(str).str.zfill(2))

# ---- 2. Z-scores WITHIN each city-year ------------------------------
grp = monthly.groupby(['City', 'Year'])
monthly['months_in_year'] = grp['Month'].transform('size')

z_cols = []
for gas in POLLUTANTS:
    zc = f'Z_{gas}'
    g  = grp[gas]
    sd = g.transform('std', ddof=0)
    monthly[zc] = (monthly[gas] - g.transform('mean')) / sd.replace(0, np.nan)
    z_cols.append(zc)

# ---- 3. MPSI, N_elevated, three conditions --------------------------
monthly['MPSI']         = monthly[z_cols].mean(axis=1)
monthly['N_elevated']   = (monthly[z_cols] >= ELEVATED_THRESHOLD).sum(axis=1)
monthly['cond1_MPSI']   = monthly['MPSI'] >= MPSI_THRESHOLD
monthly['cond2_Nelev']  = monthly['N_elevated'] >= MIN_ELEVATED
monthly['cond3_window'] = monthly['Month'].isin(WINTER_MONTHS)

monthly['SmogMonth'] = (monthly['cond1_MPSI'] &
                        monthly['cond2_Nelev'] &
                        monthly['cond3_window']).astype(int)
monthly['Formula detected Smog'] = np.where(monthly['SmogMonth'] == 1, 'Yes', 'No')
monthly['short_year_flag'] = monthly['months_in_year'] < MIN_MONTHS_PER_YEAR

smog_result = (monthly[['City', 'Country', 'date', 'Year', 'Month',
                        'MPSI', 'N_elevated', 'cond1_MPSI', 'cond2_Nelev',
                        'cond3_window', 'SmogMonth', 'Formula detected Smog',
                        'months_in_year', 'short_year_flag']]
               .sort_values(['City', 'date'])
               .reset_index(drop=True))

print(f"cities: {smog_result['City'].nunique()} | months: {len(smog_result)} | "
      f"range: {smog_result['date'].min()} -> {smog_result['date'].max()} | "
      f"SmogMonth=1: {smog_result['SmogMonth'].sum()} "
      f"({smog_result['SmogMonth'].mean()*100:.1f}%)")
print(f"months in short years (<{MIN_MONTHS_PER_YEAR} obs): "
      f"{smog_result['short_year_flag'].sum()}")

display(smog_result[['City', 'date', 'Formula detected Smog']])


In [ ]:
smog_result.to_csv('../Results/smog_month_detection_v2.csv', index=False)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

GAS_COLORS = {'NO2': '#2a78d6', 'CO': '#eb6834', 'SO2': '#1baf7a',
              'O3': '#eda100', 'AerosolIndex': '#e87ba4'}
INK, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#e5e5e2', '#ffffff'
DIVERGING = LinearSegmentedColormap.from_list(
    'mpsi', ['#104281', '#2a78d6', '#9ec5f4', '#f0efec', '#e34948', '#8c2020'])

ca = monthly.copy()
c_cols = []
for gas in POLLUTANTS:                       # each gas's contribution to MPSI = Z / 5
    ca[f'c_{gas}'] = ca[f'Z_{gas}'] / len(POLLUTANTS)
    c_cols.append(f'c_{gas}')

winter = ca[ca['Month'].isin(WINTER_MONTHS)].copy()
print(f"{ca['City'].nunique()} cities | winter months: {len(winter)} | "
      f"detected: {int(winter['SmogMonth'].sum())}")


In [ ]:
piv_m = winter.pivot_table(index='City', columns='date', values='MPSI')
piv_d = winter.pivot_table(index='City', columns='date', values='SmogMonth')
order = piv_d.sum(axis=1).sort_values(ascending=True).index      # most smog on top
piv_m, piv_d = piv_m.loc[order], piv_d.loc[order]

vmax = float(np.nanmax(np.abs(piv_m.values)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
cmap = DIVERGING.copy(); cmap.set_bad('#fafaf8')

fig, ax = plt.subplots(figsize=(max(12, 0.32 * piv_m.shape[1]),
                                0.42 * piv_m.shape[0] + 2.6), facecolor=SURFACE)
im = ax.imshow(np.ma.masked_invalid(piv_m.values), aspect='auto',
               cmap=cmap, norm=norm, interpolation='nearest')

ys, xs = np.where(piv_d.fillna(0).values == 1)                   # detection markers
ax.plot(xs, ys, 'o', ms=6, color=INK, mec='white', mew=1.2,
        linestyle='none', label='SmogMonth = 1', zorder=4)

ax.set_xticks(np.arange(piv_m.shape[1]))
ax.set_xticklabels(piv_m.columns, rotation=90, fontsize=7, color=MUTED)
ax.set_yticks(np.arange(piv_m.shape[0]))
ax.set_yticklabels(piv_m.index, fontsize=9, color=INK)
ax.set_xticks(np.arange(-.5, piv_m.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, piv_m.shape[0], 1), minor=True)
ax.grid(which='minor', color=SURFACE, linewidth=1.5)
ax.tick_params(which='both', length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title('Detected smog months by city — Oct–Feb window\n'
             'cell = MPSI (red above city-year norm, blue below); dot = all three conditions met',
             color=INK, fontsize=13, loc='left', pad=14)
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 1.02),
          fontsize=9, labelcolor=MUTED, handletextpad=0.4)

cb = fig.colorbar(im, ax=ax, pad=0.012, fraction=0.018)
cb.set_label('MPSI', color=MUTED, fontsize=9)
cb.ax.tick_params(colors=MUTED, length=0); cb.outline.set_visible(False)
cb.ax.axhline(MPSI_THRESHOLD, color=INK, lw=1.4)                 # threshold on the scale

plt.tight_layout()
plt.savefig('../Results/Figures/AllCities_smog_months_heatmap.png', dpi=200,
            facecolor=SURFACE, bbox_inches='tight')
plt.show()

display(piv_d.sum(axis=1).sort_values(ascending=False)
             .rename('detected_smog_months').to_frame())


In [ ]:
BASIS = 'detected'          # 'detected' = only SmogMonth==1 months; 'winter' = all Oct-Feb

src = winter[winter['SmogMonth'] == 1] if BASIS == 'detected' else winter
n_by_city = src.groupby('City').size()
gas_mean  = src.groupby('City')[c_cols].mean()
gas_mean.columns = POLLUTANTS
mpsi_mean = src.groupby('City')['MPSI'].mean()

zero = sorted(set(winter['City'].unique()) - set(gas_mean.index))
if zero:
    print(f"no detected smog months (excluded from this plot): {zero}")

order2 = mpsi_mean.sort_values().index
gas_mean, mpsi_mean, n_by_city = (gas_mean.loc[order2], mpsi_mean.loc[order2],
                                  n_by_city.loc[order2])
y = np.arange(len(order2))

fig, ax = plt.subplots(figsize=(11, 0.52 * len(order2) + 2.4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

pos_l = np.zeros(len(order2)); neg_l = np.zeros(len(order2))
for gas in POLLUTANTS:
    v   = gas_mean[gas].values
    pos = np.where(v > 0, v, np.nan); neg = np.where(v < 0, v, np.nan)
    ax.barh(y, pos, left=pos_l, height=0.62, color=GAS_COLORS[gas],
            edgecolor=SURFACE, linewidth=1.4, label=gas, zorder=2)
    ax.barh(y, neg, left=neg_l, height=0.62, color=GAS_COLORS[gas],
            edgecolor=SURFACE, linewidth=1.4, zorder=2)
    pos_l += np.nan_to_num(pos); neg_l += np.nan_to_num(neg)

ax.plot(mpsi_mean.values, y, 'o', ms=8, color=INK, mec=SURFACE, mew=1.5,
        linestyle='none', label='mean MPSI', zorder=5)

for i, (v, n) in enumerate(zip(mpsi_mean.values, n_by_city.values)):
    ax.annotate(f'{v:.2f}  (n={n})', xy=(pos_l[i], i), xytext=(6, 0),
                textcoords='offset points', va='center', fontsize=8, color=MUTED)

ax.axvline(0, color=MUTED, lw=1, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(order2, fontsize=10, color=INK)
ax.set_xlabel('mean contribution to MPSI  (Z / 5)', color=MUTED, fontsize=10)
ax.set_title(f'Gas contributions during {"detected smog months" if BASIS == "detected" else "Oct–Feb months"}\n'
             'segments sum to the mean MPSI dot; n = months averaged',
             color=INK, fontsize=13, loc='left', pad=14)
ax.grid(axis='x', color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
for s in ('top', 'right', 'bottom'):
    ax.spines[s].set_visible(False)
ax.spines['left'].set_color(GRID)
ax.tick_params(colors=MUTED, length=0)
ax.legend(ncol=6, frameon=False, loc='upper left', bbox_to_anchor=(0, 1.02),
          fontsize=9, labelcolor=MUTED)
ax.margins(x=0.16)

plt.tight_layout()
plt.savefig(f'../Results/Figures/AllCities_gas_contributions_{BASIS}.png', dpi=200,
            facecolor=SURFACE, bbox_inches='tight')
plt.show()

display(gas_mean.assign(mean_MPSI=mpsi_mean, n_months=n_by_city)
                .sort_values('mean_MPSI', ascending=False).round(3))


In [ ]:
def plot_city_temporal(city, mode='winter', save=True):
    """mode='winter' -> Oct-Feb months only; mode='all' -> every month, winter shaded."""
    cd = ca[ca['City'] == city].sort_values('date')
    if mode == 'winter':
        cd = cd[cd['Month'].isin(WINTER_MONTHS)]
    if cd.empty:
        print(f'{city}: no data'); return

    gc  = cd[c_cols].copy(); gc.columns = POLLUTANTS
    mp  = cd['MPSI'].values
    det = cd['SmogMonth'].values
    x   = np.arange(len(cd))

    # break the MPSI line wherever months are not consecutive
    ordm   = cd['Year'].values * 12 + cd['Month'].values
    line_y = mp.astype(float).copy()
    line_y[np.r_[False, np.diff(ordm) != 1]] = np.nan

    fig, ax = plt.subplots(figsize=(max(11, 0.30 * len(cd)), 5.5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    if mode == 'all':                                   # shade the eligible window
        for i, m in enumerate(cd['Month'].values):
            if m in WINTER_MONTHS:
                ax.axvspan(i - .5, i + .5, color='#f2f1ee', zorder=0)

    pos_b = np.zeros(len(cd)); neg_b = np.zeros(len(cd))
    for gas in POLLUTANTS:
        v   = gc[gas].values
        pos = np.where(v > 0, v, np.nan); neg = np.where(v < 0, v, np.nan)
        ax.bar(x, pos, bottom=pos_b, width=0.62, color=GAS_COLORS[gas],
               edgecolor=SURFACE, linewidth=1.4, label=gas, zorder=2)
        ax.bar(x, neg, bottom=neg_b, width=0.62, color=GAS_COLORS[gas],
               edgecolor=SURFACE, linewidth=1.4, zorder=2)
        pos_b += np.nan_to_num(pos); neg_b += np.nan_to_num(neg)



    ax.axhline(0, color=MUTED, lw=1, zorder=3)
    ax.axhline(MPSI_THRESHOLD, color=MUTED, lw=1.2, ls='--', zorder=3)
    ax.annotate('MPSI ≥ 1', xy=(len(cd) - 0.4, MPSI_THRESHOLD), xytext=(0, 4),
                textcoords='offset points', ha='right', va='bottom',
                color=MUTED, fontsize=9)

    for i, (v, hit) in enumerate(zip(mp, det)):          # label detections only
        if hit:
            ax.annotate(f' ', xy=(i, v), xytext=(0, 11),
                        textcoords='offset points', ha='center', va='bottom',
                        fontsize=8, fontweight='bold', color=INK, linespacing=1.2)

    step = 1 if mode == 'winter' else 2
    ax.set_xticks(x[::step])
    ax.set_xticklabels(cd['date'].values[::step], rotation=90, fontsize=8, color=MUTED)
    ax.set_ylabel('contribution to MPSI  (Z / 5)', color=MUTED, fontsize=10)
    ax.set_title(f'{city}',
                 color=INK, fontsize=13, loc='left', pad=14)
    ax.grid(axis='y', color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    ax.legend(ncol=6, frameon=False, loc='upper left', bbox_to_anchor=(0, 1.02),
              fontsize=9, labelcolor=MUTED)

    plt.tight_layout()
    if save:
        plt.savefig(f'../Results/Figures/{city}_MPSI_temporal_{mode}.png', dpi=200,
                    facecolor=SURFACE, bbox_inches='tight')
    plt.show()


for city in sorted(ca['City'].unique()):
    plot_city_temporal(city, mode='winter')


In [ ]:
TEXT = '#000000'          # jet black for all written text

def plot_city_temporal(city, mode='winter', save=True):
    """mode='winter' -> Oct-Feb months only; mode='all' -> every month, winter shaded."""
    cd = ca[ca['City'] == city].sort_values('date')
    if mode == 'winter':
        cd = cd[cd['Month'].isin(WINTER_MONTHS)]
    if cd.empty:
        print(f'{city}: no data'); return

    gc  = cd[c_cols].copy(); gc.columns = POLLUTANTS
    mp  = cd['MPSI'].values
    det = cd['SmogMonth'].values
    x   = np.arange(len(cd))

    # break the MPSI line wherever months are not consecutive
    ordm   = cd['Year'].values * 12 + cd['Month'].values
    line_y = mp.astype(float).copy()
    line_y[np.r_[False, np.diff(ordm) != 1]] = np.nan

    fig, ax = plt.subplots(figsize=(max(11, 0.30 * len(cd)), 5.5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    if mode == 'all':                                   # shade the eligible window
        for i, m in enumerate(cd['Month'].values):
            if m in WINTER_MONTHS:
                ax.axvspan(i - .5, i + .5, color='#f2f1ee', zorder=0)

    pos_b = np.zeros(len(cd)); neg_b = np.zeros(len(cd))
    for gas in POLLUTANTS:
        v   = gc[gas].values
        pos = np.where(v > 0, v, np.nan); neg = np.where(v < 0, v, np.nan)
        ax.bar(x, pos, bottom=pos_b, width=0.62, color=GAS_COLORS[gas],
               edgecolor=SURFACE, linewidth=1.4, label=gas, zorder=2)
        ax.bar(x, neg, bottom=neg_b, width=0.62, color=GAS_COLORS[gas],
               edgecolor=SURFACE, linewidth=1.4, zorder=2)
        pos_b += np.nan_to_num(pos); neg_b += np.nan_to_num(neg)



    ax.axhline(0, color=MUTED, lw=1, zorder=3)
    ax.axhline(MPSI_THRESHOLD, color=MUTED, lw=1.2, ls='--', zorder=3)
    ax.annotate(' ', xy=(len(cd) - 0.4, MPSI_THRESHOLD), xytext=(0, 4),
                textcoords='offset points', ha='right', va='bottom',
                color=TEXT, fontsize=12, fontweight='bold')

    for i, (v, hit) in enumerate(zip(mp, det)):          # label detections only
        if hit:
            ax.annotate(f' ', xy=(i, v), xytext=(0, 11),
                        textcoords='offset points', ha='center', va='bottom',
                        fontsize=11, fontweight='bold', color=TEXT, linespacing=1.2)

    step = 1 if mode == 'winter' else 2
    ax.set_xticks(x[::step])
    ax.set_xticklabels(cd['date'].values[::step], rotation=90, fontsize=11, color=TEXT)
    ax.set_ylabel('contribution to MPSI  (Z / 5)', color=TEXT, fontsize=13)
    ax.set_title(f'{city}',
                 color=TEXT, fontsize=18, fontweight='bold', loc='left', pad=14)
    ax.grid(axis='y', color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.tick_params(colors=TEXT, labelsize=11, length=0)
    ax.legend(ncol=6, frameon=False, loc='upper left', bbox_to_anchor=(0, 1.02),
              fontsize=12, labelcolor=TEXT)

    plt.tight_layout()
    if save:
        plt.savefig(f'../Results/Figures/{city}_MPSI_temporal_{mode}.png', dpi=200,
                    facecolor=SURFACE, bbox_inches='tight')
    plt.show()


for city in sorted(ca['City'].unique()):
    plot_city_temporal(city, mode='winter')


FIND GASES WEIGHTS 

In [ ]:
import numpy as np, pandas as pd

FIT_ON = 'all'      # 'all' = every month; 'winter' = Oct-Feb only

fit = monthly if FIT_ON == 'all' else monthly[monthly['Month'].isin(WINTER_MONTHS)]
Z   = fit[z_cols].dropna()
Z.columns = POLLUTANTS
n, k = Z.shape
print(f'fitting weights on {n} city-months, {k} gases  (FIT_ON={FIT_ON})')

# ---- 1. PCA (first principal component of the correlation matrix) ----
R = np.corrcoef(Z.values, rowvar=False)
eigval, eigvec = np.linalg.eigh(R)
idx = np.argsort(eigval)[::-1]
eigval, eigvec = eigval[idx], eigvec[:, idx]
pc1 = eigvec[:, 0]
pc1 = pc1 * np.sign(pc1.sum())                     # orient so the bulk loads positive
w_pca = np.abs(pc1) / np.abs(pc1).sum()
print(f'PC1 explains {eigval[0]/eigval.sum()*100:.1f}% of variance')

# ---- 2. Entropy weight method (EWM) ----------------------------------
X = (Z - Z.min()) / (Z.max() - Z.min())            # min-max to [0,1]
P = X / X.sum(axis=0)
E = -(1 / np.log(n)) * np.where(P > 0, P * np.log(P.where(P > 0, 1)), 0).sum(axis=0)
D = 1 - E                                          # divergence
w_ent = D / D.sum()

# ---- 3. CRITIC (contrast intensity x conflict) -----------------------
sd   = X.std(ddof=0)
conf = (1 - X.corr()).sum(axis=0)
C    = sd * conf
w_cri = C / C.sum()

# ---- 4. equal weights (the formula as specified) ---------------------
w_eq = pd.Series(1 / k, index=POLLUTANTS)

weights = pd.DataFrame({'equal': w_eq,
                        'PCA'    : np.asarray(w_pca),
                        'entropy': np.asarray(w_ent),
                        'CRITIC' : np.asarray(w_cri)}, index=POLLUTANTS)
weights.loc['SUM'] = weights.sum()
print('--- derived weights ---')
display(weights.round(4))

# diagnostics that explain WHY each method landed where it did
diag = pd.DataFrame({
    'PC1_loading'          : np.asarray(pc1),
    'entropy_E'            : np.asarray(E),
    'CRITIC_sd'            : np.asarray(sd),
    'CRITIC_conflict'      : np.asarray(conf),
    'mean_corr_with_others': (R.sum(axis=0) - 1) / (k - 1),
}, index=POLLUTANTS)
display(diag.round(3))


In [ ]:
# ---- re-score MPSI under each weight vector --------------------------
cmp = monthly[['City', 'date', 'Year', 'Month', 'N_elevated', 'cond2_Nelev',
               'cond3_window']].copy()

for scheme in ['equal', 'PCA', 'entropy', 'CRITIC']:
    w = weights.loc[POLLUTANTS, scheme].values
    mpsi = (monthly[z_cols].values * w).sum(axis=1)          # weights sum to 1
    cmp[f'MPSI_{scheme}']  = mpsi
    cmp[f'Smog_{scheme}']  = ((mpsi >= MPSI_THRESHOLD) &
                              cmp['cond2_Nelev'] & cmp['cond3_window']).astype(int)

summary = pd.DataFrame({
    'smog_months': {s: int(cmp[f'Smog_{s}'].sum()) for s in
                    ['equal', 'PCA', 'entropy', 'CRITIC']},
    'mean_MPSI_winter': {s: cmp.loc[cmp['cond3_window'], f'MPSI_{s}'].mean()
                         for s in ['equal', 'PCA', 'entropy', 'CRITIC']},
})
print('--- effect of the weighting choice ---')
display(summary.round(3))

# agreement with the as-specified equal-weight labels
for s in ['PCA', 'entropy', 'CRITIC']:
    agree = (cmp['Smog_equal'] == cmp[f'Smog_{s}']).mean()
    flip  = (cmp['Smog_equal'] != cmp[f'Smog_{s}']).sum()
    print(f'{s:8s}: {agree*100:5.1f}% same label as equal weights  ({flip} months flip)')

# per-city detection counts under each scheme
display(cmp.groupby('City')[[f'Smog_{s}' for s in
                             ['equal', 'PCA', 'entropy', 'CRITIC']]].sum())

cmp.to_csv('../Results/smog_month_weight_comparison.csv', index=False)
weights.to_csv('../Results/MPSI_gas_weights.csv')


LOGISTIC REGRESSION

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

fitdf  = monthly.dropna(subset=z_cols)
X      = fitdf[z_cols].values
y      = fitdf['Month'].isin(WINTER_MONTHS).astype(int).values   # label NOT from the formula
groups = fitdf['City'].values

lr = LogisticRegression(C=np.inf, max_iter=5000, class_weight='balanced').fit(X, y)
coef = lr.coef_[0]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
p  = cross_val_predict(lr, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]
print(f'AUC  in-sample {roc_auc_score(y, lr.predict_proba(X)[:,1]):.3f} | '
      f'grouped CV {roc_auc_score(y, p):.3f}')

w        = coef / np.abs(coef).sum()          # signed, |w| sums to 1
raw      = fitdf[z_cols].values @ w
sigma_w  = raw.std(ddof=0)
w_final  = w / sigma_w                        # -> MPSI* has unit variance

print(f'sigma_w = {sigma_w:.4f}')
display(pd.DataFrame({'coefficient': coef,
                      'weight_share': np.abs(coef)/np.abs(coef).sum(),
                      'w_final': w_final}, index=POLLUTANTS).round(4))

# ---- apply the refined equation -------------------------------------
TAU = 1.8      # 1.8 keeps the original strictness in SD units (orig SD = 0.557)

monthly['MPSI_star'] = monthly[z_cols].values @ w_final
monthly['SmogMonth_refined'] = ((monthly['MPSI_star'] >= TAU) &
                                monthly['cond2_Nelev'] &
                                monthly['cond3_window']).astype(int)
monthly['Refined detected Smog'] = np.where(monthly['SmogMonth_refined'] == 1, 'Yes', 'No')

print(f"equal-weight: {int(monthly['SmogMonth'].sum())} | "
      f"refined (tau={TAU}): {int(monthly['SmogMonth_refined'].sum())}")
display(monthly.groupby('City')[['SmogMonth', 'SmogMonth_refined']].sum())

monthly[['City','Country','date','Year','Month','MPSI','MPSI_star','N_elevated',
         'SmogMonth','SmogMonth_refined','Formula detected Smog',
         'Refined detected Smog']].to_csv('../Results/smog_month_refined.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt

TEXT, MUTED, GRID, SURFACE = '#000000', '#52514e', '#e5e5e2', '#ffffff'
UP, DOWN = '#d03b3b', '#2a78d6'          # raises the index / lowers it

share = np.abs(coef) / np.abs(coef).sum()
wtab  = (pd.DataFrame({'w_final': w_final, 'share': share, 'coef': coef},
                      index=POLLUTANTS)
           .sort_values('w_final'))
seas  = monthly.groupby(monthly['Month'].isin(WINTER_MONTHS))[z_cols].mean()
seas.columns = POLLUTANTS

fig, axes = plt.subplots(2, 2, figsize=(15.5, 11.5), facecolor=SURFACE)
fig.suptitle('What the data says each gas is worth in the smog index',
             color=TEXT, fontsize=21, fontweight='bold', x=0.008, ha='left', y=0.995)

# ---- A. the weights themselves --------------------------------------
ax = axes[0, 0]; ax.set_facecolor(SURFACE)
y  = np.arange(len(wtab))
ax.barh(y, wtab['w_final'], height=0.6, zorder=2,
        color=[UP if v > 0 else DOWN for v in wtab['w_final']],
        edgecolor=SURFACE, linewidth=1.4)
for i, v in enumerate(wtab['w_final']):
    ax.annotate(f'{v:+.2f}', xy=(v, i), xytext=(7 if v > 0 else -7, 0),
                textcoords='offset points', va='center',
                ha='left' if v > 0 else 'right',
                fontsize=13, fontweight='bold', color=TEXT)
ax.axvline(0, color=TEXT, lw=1.2, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(wtab.index, fontsize=14, color=TEXT)
ax.set_title('A.  Which gases signal smog, and which argue against it\n'
             'bar right = a high reading means smog · bar left = a high reading means NO smog',
             fontsize=14, fontweight='bold', color=TEXT, loc='left', pad=12)
ax.set_xlabel('weight in the refined equation', fontsize=12, color=TEXT)
ax.margins(x=0.22)

# ---- B. assumed vs measured importance ------------------------------
ax = axes[0, 1]; ax.set_facecolor(SURFACE)
o  = np.abs(wtab['share']).sort_values().index
y  = np.arange(len(o))
ax.barh(y - 0.19, [1 / len(POLLUTANTS)] * len(o), height=0.34, color='#b9b8b2',
        edgecolor=SURFACE, linewidth=1.4, label='original formula: everyone equal (20%)', zorder=2)
ax.barh(y + 0.19, share[[POLLUTANTS.index(g) for g in o]], height=0.34,
        color='#4a3aa7', edgecolor=SURFACE, linewidth=1.4,
        label='measured importance', zorder=2)
for i, g in enumerate(o):
    ax.annotate(f'{share[POLLUTANTS.index(g)]*100:.0f}%', xy=(share[POLLUTANTS.index(g)], i + 0.19),
                xytext=(6, 0), textcoords='offset points', va='center',
                fontsize=12, fontweight='bold', color=TEXT)
ax.set_yticks(y); ax.set_yticklabels(o, fontsize=14, color=TEXT)
ax.set_title('B.  The formula assumed all five gases matter equally\n'
             'SO₂ actually carries three times the weight of NO₂',
             fontsize=14, fontweight='bold', color=TEXT, loc='left', pad=12)
ax.set_xlabel('share of the total signal', fontsize=12, color=TEXT)
ax.legend(frameon=False, fontsize=11, labelcolor=TEXT, loc='lower right')
ax.margins(x=0.18)

# ---- C. why the signs came out that way -----------------------------
ax = axes[1, 0]; ax.set_facecolor(SURFACE)
y  = np.arange(len(POLLUTANTS))
for i, g in enumerate(POLLUTANTS):
    a, b = seas.loc[False, g], seas.loc[True, g]
    ax.plot([a, b], [i, i], '-', lw=2.5, color=GRID, zorder=1)
    ax.plot(a, i, 'o', ms=13, color='#eda100', mec=SURFACE, mew=1.5, zorder=3,
            label='Mar–Sep (rest of year)' if i == 0 else None)
    ax.plot(b, i, 'o', ms=13, color=DOWN, mec=SURFACE, mew=1.5, zorder=3,
            label='Oct–Feb (smog season)' if i == 0 else None)
ax.axvline(0, color=TEXT, lw=1.2, zorder=2)
ax.set_yticks(y); ax.set_yticklabels(POLLUTANTS, fontsize=14, color=TEXT)
ax.set_title('C.  Why O₃ and UVAI get negative weights\n'
             'SO₂, CO, NO₂ rise in smog season — O₃ and UVAI fall',
             fontsize=14, fontweight='bold', color=TEXT, loc='left', pad=12)
ax.set_xlabel('average level vs the city’s own yearly norm (z-score)', fontsize=12, color=TEXT)
ax.legend(frameon=False, fontsize=12, labelcolor=TEXT, loc='lower right')
ax.margins(x=0.22)

# ---- D. what it changes ---------------------------------------------
ax = axes[1, 1]; ax.set_facecolor(SURFACE)
cnt = (monthly.groupby('City')[['SmogMonth', 'SmogMonth_refined']].sum()
              .sort_values('SmogMonth_refined'))
y = np.arange(len(cnt))
ax.barh(y - 0.19, cnt['SmogMonth'], height=0.34, color='#b9b8b2',
        edgecolor=SURFACE, linewidth=1.4, label='original equation', zorder=2)
ax.barh(y + 0.19, cnt['SmogMonth_refined'], height=0.34, color='#1baf7a',
        edgecolor=SURFACE, linewidth=1.4, label='refined equation', zorder=2)
ax.set_yticks(y); ax.set_yticklabels(cnt.index, fontsize=13, color=TEXT)
ax.set_title(f'D.  Smog months found in each city\n'
             f'refined equation (τ={TAU}) recovers cities the original missed entirely',
             fontsize=14, fontweight='bold', color=TEXT, loc='left', pad=12)
ax.set_xlabel('number of smog months detected', fontsize=12, color=TEXT)
ax.legend(frameon=False, fontsize=12, labelcolor=TEXT, loc='lower right')
ax.margins(x=0.12)

for ax in axes.ravel():
    ax.grid(axis='x', color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.tick_params(colors=TEXT, labelsize=12, length=0)

plt.tight_layout(rect=[0, 0, 1, 0.975])
plt.savefig('../Results/Figures/MPSI_weights_explained.png', dpi=200,
            facecolor=SURFACE, bbox_inches='tight')
plt.show()


DELHJI  ANALYSIS

In [ ]:
CITY = 'Delhi'

d = monthly[monthly['City'] == CITY].sort_values('date').set_index('date')

z = d[z_cols].copy(); z.columns = POLLUTANTS
contrib = z / len(POLLUTANTS)                      # each gas's share of MPSI

detail = pd.concat([
    d[POLLUTANTS].add_suffix('_raw'),
    z.add_prefix('Z_').round(3),
    contrib.add_prefix('contrib_').round(3),
    d[['MPSI', 'N_elevated', 'cond1_MPSI', 'cond2_Nelev', 'cond3_window',
       'SmogMonth', 'Formula detected Smog', 'Year', 'months_in_year']],
], axis=1)
detail['MPSI'] = detail['MPSI'].round(3)

print(f'{CITY}: {len(detail)} months, {detail["date"].min() if "date" in detail else detail.index.min()} '
      f'-> {detail.index.max()} | SmogMonth=1: {int(detail["SmogMonth"].sum())}')
display(detail)



In [ ]:
win_mask = d['Month'].isin(WINTER_MONTHS)

drivers = pd.DataFrame({
    'mean_Z_all_months' : z.mean(),
    'mean_Z_Oct_Feb'    : z[win_mask.values].mean(),
    'mean_Z_Mar_Sep'    : z[~win_mask.values].mean(),
    'mean_contrib_to_MPSI_OctFeb': contrib[win_mask.values].mean(),
    'elevated_rate_OctFeb' : (z[win_mask.values] >= ELEVATED_THRESHOLD).mean(),
    'elevated_rate_MarSep' : (z[~win_mask.values] >= ELEVATED_THRESHOLD).mean(),
}).sort_values('mean_contrib_to_MPSI_OctFeb', ascending=False)
print('--- gas roles (positive contrib = pushes MPSI up) ---')
display(drivers.round(3))

# top / bottom driver gas for every winter month
w = contrib[win_mask.values].copy()
w['MPSI']       = d.loc[win_mask.values, 'MPSI'].round(3)
w['top_driver'] = contrib[win_mask.values].idxmax(axis=1)
w['biggest_drag']  = contrib[win_mask.values].idxmin(axis=1)
w['verdict']    = d.loc[win_mask.values, 'Formula detected Smog']
print('--- Oct-Feb months: contributions, driver, drag, verdict ---')
display(w.round(3))

# how often each gas is the top driver of a DETECTED month
hits = w[w['verdict'] == 'Yes']
print(f'detected smog months: {len(hits)}')
display(hits['top_driver'].value_counts().rename('times_top_driver'))

# per-year summary
display(d.groupby('Year').agg(months='MPSI', mean_MPSI=('MPSI', 'mean'),
                              max_MPSI=('MPSI', 'max'),
                              smog_months=('SmogMonth', 'sum')).round(3)
         if False else
        d.groupby('Year').agg(mean_MPSI=('MPSI', 'mean'),
                              max_MPSI=('MPSI', 'max'),
                              smog_months=('SmogMonth', 'sum'),
                              n_months=('MPSI', 'size')).round(3))


In [ ]:
import matplotlib.pyplot as plt

GAS_COLORS = {'NO2': '#2a78d6', 'CO': '#eb6834', 'SO2': '#1baf7a',
              'O3': '#eda100', 'AerosolIndex': '#e87ba4'}
INK, MUTED, GRID, SURFACE = '#0b0b0b', '#52514e', '#e5e5e2', '#ffffff'

wc  = contrib[win_mask.values]
mp  = d.loc[win_mask.values, 'MPSI']
det = d.loc[win_mask.values, 'SmogMonth'].values
x   = np.arange(len(wc))

fig, ax = plt.subplots(figsize=(13, 5.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

pos_bot = np.zeros(len(wc)); neg_bot = np.zeros(len(wc))
for gas in POLLUTANTS:
    v   = wc[gas].values
    pos = np.where(v > 0, v, np.nan)
    neg = np.where(v < 0, v, np.nan)
    ax.bar(x, pos, bottom=pos_bot, width=0.62, color=GAS_COLORS[gas],
           edgecolor=SURFACE, linewidth=1.4, label=gas, zorder=2)
    ax.bar(x, neg, bottom=neg_bot, width=0.62, color=GAS_COLORS[gas],
           edgecolor=SURFACE, linewidth=1.4, zorder=2)
    pos_bot += np.nan_to_num(pos); neg_bot += np.nan_to_num(neg)

# break the line where months are not consecutive (i.e. between smog seasons)
ordm = (d.loc[win_mask.values, 'Year'].values * 12 +
        d.loc[win_mask.values, 'Month'].values)
line_y = mp.values.astype(float).copy()
line_y[np.r_[False, np.diff(ordm) != 1]] = np.nan   # NaN starts a new segment

ax.plot(x, line_y, '-', lw=2, color=INK, zorder=4, solid_capstyle='round')
ax.plot(x, mp.values, 'o', ms=8, color=INK, label='MPSI (net)',
        markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=5,
        linestyle='none')


ax.axhline(0, color=MUTED, lw=1, zorder=3)
ax.axhline(MPSI_THRESHOLD, color=MUTED, lw=1.2, ls='--', zorder=3)
ax.annotate('MPSI ≥ 1 threshold', xy=(len(wc) - 0.4, MPSI_THRESHOLD),
            xytext=(0, 4), textcoords='offset points',
            ha='right', va='bottom', color=MUTED, fontsize=9)

for i, (v, hit) in enumerate(zip(mp.values, det)):        # label detected months only
    if hit:
        ax.annotate(f'', xy=(i, v), xytext=(0, 11),
                    textcoords='offset points', ha='center', va='bottom',
                    fontsize=8, fontweight='bold', color=INK, linespacing=1.2)

ax.set_xticks(x)
ax.set_xticklabels(wc.index, rotation=90, fontsize=8, color=MUTED)
ax.set_ylabel('contribution to MPSI  (Z / 5)', color=MUTED, fontsize=10)
ax.set_title(f'{CITY}',
             color=INK, fontsize=13, loc='left', pad=14)
ax.grid(axis='y', color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color(GRID)
ax.tick_params(colors=MUTED, length=0)
ax.legend(ncol=6, frameon=False, loc='upper left', bbox_to_anchor=(0, 1.02),
          fontsize=9, labelcolor=MUTED)

plt.tight_layout()
plt.savefig(f'../Results/Figures/{CITY}_MPSI_contributions.png', dpi=200,
            facecolor=SURFACE, bbox_inches='tight')
plt.show()
